# SSE Migration Guide: htmx v2 → v4 for FastHTML

Covers migrating FastHTML SSE apps from htmx v2 to v4, including:
- Full htmx v4 SSE extension reference and event name changes
- SSE htmx v2 document
- SSE in fasthtml with htmx v2

## SSE htmx v4 doc
---
title: "Server-Sent Events (SSE)"
description: "Stream server updates using Server-Sent Events"
keywords: ["sse", "server-sent events", "event stream", "streaming", "real-time"]
---

The SSE extension adds support for [Server-Sent Events](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events) streaming to htmx. It works by intercepting any htmx response with `Content-Type: text/event-stream` and streaming SSE messages into the DOM in real-time.

SSE is a lightweight alternative to WebSockets that works over existing HTTP connections, making it easy to use through proxy servers and firewalls. SSE is uni-directional: the server pushes data to the client. If you need bi-directional communication, consider [WebSockets](/docs/extensions/ws) instead.

## Installing

Include the extension script after htmx:

```html
<script src="/path/to/htmx.js"></script>
<script src="/path/to/ext/hx-sse.js"></script>
```

## How It Works

The SSE extension hooks into htmx's request pipeline. When any htmx request receives a response with `Content-Type: text/event-stream`, the extension takes over and streams SSE messages into the DOM instead of performing a normal swap.

This means **any [`hx-get`](/reference/attributes/hx-get), [`hx-post`](/reference/attributes/hx-post), etc. that returns an SSE stream will just work**, no special attributes needed beyond loading the extension.

## `hx-sse:connect`

For persistent SSE connections (auto-connect on load, reconnect on failure), use `hx-sse:connect`:

```html
<!-- Auto-connects on load, streams messages into the div -->
<div hx-sse:connect="/stream">
    Waiting for messages...
</div>
```

`hx-sse:connect` is convenience sugar for a well-preconfigured `hx-get`. It defaults to:
- **Trigger**: `load` (connects immediately)
- **Reconnect**: enabled with exponential backoff
- **Pause on background**: closes the stream when the tab is backgrounded, reconnects when visible

### Using with Standard Attributes

`hx-sse:connect` works with all standard htmx attributes:

```html
<!-- Swap into a different target -->
<button hx-sse:connect="/notifications" hx-target="#alerts">
    Start Notifications
</button>
<div id="alerts"></div>

<!-- Append messages instead of replacing -->
<div hx-sse:connect="/log" hx-swap="beforeend">
    <h3>Log:</h3>
</div>
```

### Trigger Modifiers

All standard [`hx-trigger`](/reference/attributes/hx-trigger) modifiers are supported:

```html
<!-- Connect after a delay -->
<div hx-sse:connect="/stream" hx-trigger="load delay:2s">

<!-- Connect on click -->
<button hx-sse:connect="/stream" hx-trigger="click">Start</button>

<!-- Connect on click, only once -->
<button hx-sse:connect="/stream" hx-trigger="click once">Start</button>
```

### Using Standard htmx Attributes

Since the extension intercepts based on Content-Type, any htmx request that returns `text/event-stream` will be streamed automatically:

```html
<!-- hx-get, hx-post, etc. all work -->
<div hx-get="/stream" hx-trigger="load">
    Waiting...
</div>

<button hx-post="/generate" hx-target="#output">
    Generate
</button>
```

The difference is that `hx-sse:connect` enables reconnection and `pauseOnBackground` by default, while standard attributes do not.

## `hx-sse:close`

Use `hx-sse:close` to gracefully close an SSE connection when a specific named event is received from the server:

```html
<div hx-sse:connect="/stream" hx-sse:close="done">
    Streaming until server sends "done"...
</div>
```

When the server sends `event: done`, the connection is closed and an `htmx:sse:close` event is fired with `detail.reason === "message"`.

## Named Events

SSE messages with an `event:` field are dispatched as DOM events on the source element rather than being swapped:

```txt
event: notification
data: {"title": "New message", "body": "Hello!"}
```

```html
<div hx-sse:connect="/events"
     hx-on:notification="alert(event.detail.data)">
</div>
```

Messages without an `event:` field are swapped into the DOM as HTML content.

## Configuration

Configure SSE behavior globally via `htmx.config.sse` or per-element via [`hx-config`](/reference/attributes/hx-config):

```html
<!-- Global config -->
<meta name="htmx-config" content='{
    "sse": {
        "reconnect": true,
        "reconnectDelay": 500,
        "reconnectMaxDelay": 60000,
        "reconnectMaxAttempts": 50,
        "reconnectJitter": 0.3,
        "pauseOnBackground": false
    }
}'>

<!-- Per-element override -->
<div hx-sse:connect="/stream" hx-config='{"sse": {"reconnect": false}}'>
```

| Option | Default (`hx-sse:connect`) | Default (`hx-get`) | Description |
|--------|---------------------------|---------------------|-------------|
| `reconnect` | `true` | `false` | Auto-reconnect on stream end |
| `reconnectDelay` | `500` | `500` | Initial reconnect delay (ms) |
| `reconnectMaxDelay` | `60000` | `60000` | Maximum reconnect delay (ms) |
| `reconnectMaxAttempts` | `Infinity` | `Infinity` | Maximum reconnection attempts |
| `reconnectJitter` | `0.3` | `0.3` | Jitter factor (0-1) for delay randomization |
| `pauseOnBackground` | `true` | `false` | Disconnect when the tab is backgrounded, reconnect when visible (see [Background Tab Behavior](#background-tab-behavior)) |

### Reconnection Strategy

The extension uses exponential backoff with jitter:

- **Formula**: `delay = min(reconnectDelay × 2^(attempt-1), reconnectMaxDelay)`
- **Jitter**: Adds ±`reconnectJitter` randomization to avoid thundering herd
- **Last-Event-ID**: Automatically sent on reconnection if the server provided message IDs (see [Background Tab Behavior](#background-tab-behavior))

### Background Tab Behavior

When `pauseOnBackground` is enabled (the default for `hx-sse:connect`), the extension disconnects the
stream when the browser tab is hidden and reconnects when the tab becomes visible again. This exists
because some browsers (notably iOS Safari) silently kill SSE connections when the app is backgrounded
without firing any error events, leaving the connection in a zombie state.

**Messages sent by the server while the tab is in the background are not received by the client.** Whether
those messages can be recovered depends on your server:

- If the server includes `id:` fields in its SSE messages, the extension tracks the last received ID and
  sends it as a `Last-Event-ID` header when reconnecting.
- If the server reads the `Last-Event-ID` header and replays missed messages, nothing is lost.
- If the server does not send `id:` fields or does not support `Last-Event-ID`, messages sent during the
  background period are lost.

#### Example: Resumable Notifications Stream

**Server** (Python with FastAPI + sse-starlette):

```python
from fastapi import FastAPI, Request
from sse_starlette.sse import EventSourceResponse

app = FastAPI()
notifications = []  # In production, use a database

@app.get("/notifications")
async def sse(request: Request):
    last_id = request.headers.get("last-event-id")

    async def stream():
        # Replay any missed messages
        start = 0
        if last_id:
            for i, n in enumerate(notifications):
                if str(n["id"]) == last_id:
                    start = i + 1
                    break
            for n in notifications[start:]:
                yield {"id": str(n["id"]), "data": n["data"]}

        # Stream new messages as they arrive
        seen = len(notifications)
        while True:
            if len(notifications) > seen:
                for n in notifications[seen:]:
                    yield {"id": str(n["id"]), "data": n["data"]}
                seen = len(notifications)
            await asyncio.sleep(0.5)

    return EventSourceResponse(stream())
```

**Client:**

```html
<div hx-sse:connect="/notifications" hx-swap="beforeend">
    <!-- Notifications appear here -->
</div>
```

When the user switches tabs and comes back, the extension reconnects with
`Last-Event-ID: <last-received-id>`, and the server replays any notifications
that were sent in the meantime.

## Events

### `htmx:before:sse:connection`

Fired before a connection attempt (initial or reconnection). Set `detail.connection.cancelled = true` to prevent the connection.

```javascript
document.body.addEventListener('htmx:before:sse:connection', function(evt) {
    if (evt.detail.connection.attempt > 10) {
        evt.detail.connection.cancelled = true;
    }
});
```

- `detail.connection.attempt` - attempt number (`0` = initial, `> 0` = reconnection)
- `detail.connection.delay` - the delay before connection (ms), modifiable
- `detail.connection.url` - the SSE endpoint URL
- `detail.connection.lastEventId` - the last event ID received
- `detail.connection.cancelled` - set to `true` to cancel

### `htmx:after:sse:connection`

Fired after a successful connection (or reconnection) to the SSE stream.

- `detail.connection.attempt` - attempt number (`0` = initial, `> 0` = reconnection)
- `detail.connection.url` - the SSE endpoint URL
- `detail.connection.status` - the HTTP status code
- `detail.connection.lastEventId` - the last event ID received

### `htmx:before:sse:message`

Fired before each SSE message is processed. All fields are modifiable.

```javascript
document.body.addEventListener('htmx:before:sse:message', function(evt) {
    // Skip heartbeats
    if (evt.detail.message.event === 'heartbeat') {
        evt.detail.message.cancelled = true;
    }

    // Transform data before swap
    evt.detail.message.data = sanitize(evt.detail.message.data);
});
```

- `detail.message.data` - the message data (modifiable)
- `detail.message.event` - the event type (modifiable)
- `detail.message.id` - the message ID (if specified)
- `detail.message.cancelled` - set to `true` to skip

### `htmx:after:sse:message`

Fired after an SSE message has been processed.

- `detail.message` - same shape as `htmx:before:sse:message`

### `htmx:sse:error`

Fired when a stream error occurs.

- `detail.error` - the error object

### `htmx:sse:close`

Fired when an SSE connection is closed.

- `detail.reason` - why the connection was closed:
  - `"message"` - closed by `hx-sse:close` matching a named event
  - `"removed"` - the element was removed from the DOM
  - `"ended"` - the stream ended naturally or reconnection was exhausted
  - `"cancelled"` - the initial connection was cancelled via `htmx:before:sse:connection`
  - `"cleanup"` - closed during element cleanup (e.g., parent swap)

## Upgrading from htmx 2.x

The htmx 2.x SSE extension (`htmx-ext-sse`) has been rewritten for htmx 4.

### What Changed

The 2.x extension was built around `EventSource` and had its own swap mechanism (`sse-swap`) that operated outside of htmx's normal request/response pipeline. The 4.x extension removes all of that. It hooks into htmx's standard request pipeline instead: any htmx request that receives a `Content-Type: text/event-stream` response is automatically streamed as SSE.

This means:
- **Unnamed messages** (no `event:` field) are swapped into the DOM using htmx's normal swap pipeline.
- **Named messages** (with an `event:` field) are dispatched as DOM events on the source element. They are not swapped.
- `sse-swap` is gone entirely. There is no equivalent, because the extension no longer has its own swap system.

### Connecting and Swapping

**htmx 2.x:**
```html
<div sse-connect="/chatroom" sse-swap="message">
    Contents of this box will be updated in real time
    with every SSE message received from the chatroom.
</div>
```

**htmx 4.x:**
```html
<div hx-sse:connect="/chatroom">
    Contents of this box will be updated in real time
    with every SSE message received from the chatroom.
</div>
```

### Event Changes

| htmx 2.x Event | htmx 4.x Event | Notes |
|-----------------|-----------------|-------|
| `htmx:sseOpen` | `htmx:after:sse:connection` | `detail.connection.attempt === 0` for initial |
| `htmx:sseError` | `htmx:sse:error` | `detail.error` contains the error |
| `htmx:sseBeforeMessage` | `htmx:before:sse:message` | Set `detail.message.cancelled = true` to skip |
| `htmx:sseMessage` | `htmx:after:sse:message` | |
| `htmx:sseClose` | `htmx:sse:close` | `detail.reason` indicates why |

### Other Changes

- **No more `EventSource`**: uses `fetch()` + `ReadableStream`, enabling POST requests, custom headers, and cookies.
- **Reconnection**: `hx-sse:connect` reconnects automatically with exponential backoff. Configure via `hx-config`.
- **Background tab handling**: pauses streams when the tab is backgrounded, reconnects when visible (configurable via `pauseOnBackground`).
- **Any HTTP method**: `hx-post`, [`hx-put`](/reference/attributes/hx-put), etc. all work with SSE responses.

## SSE htmx v2 doc


The `Server Sent Events` extension connects to
an [EventSource](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events) directly
from HTML. It manages the connections to your web server, listens for server events, and then swaps their contents into
your htmx webpage in real-time.

SSE is a lightweight alternative to WebSockets that works over existing HTTP connections, so it is easy to use through
proxy servers and firewalls. Remember, SSE is a uni-directional service, so you cannot send any messages to an SSE
server once the connection has been established. If you need bi-directional communication, then you should consider
using [WebSockets](@/extensions/ws.md) instead.

This extension replaces the experimental `hx-sse` attribute built into previous versions of htmx. For help migrating
from older versions, see the migration guide at the bottom of this page.

Use the following attributes to configure how SSE connections behave:

* `sse-connect="<url>"` - The URL of the SSE server.
* `sse-swap="<message-name>"` - The name of the message to swap into the DOM.
* `hx-trigger="sse:<message-name>"` - SSE messages can also trigger HTTP callbacks using
  the [`hx-trigger`](https://htmx.org/attributes/hx-trigger) attribute.
* `sse-close=<message-name>` - To close the EventStream gracefully when that message is received. This might be helpful
  if you want to send information to a client that will eventually stop.



## Usage

```html

<div hx-ext="sse" sse-connect="/chatroom" sse-swap="message">
    Contents of this box will be updated in real time
    with every SSE message received from the chatroom.
</div>
```

### Connecting to an SSE Server

To connect to an SSE server, use the `hx-ext="sse"` attribute to install the extension on that HTML element, then
add `sse-connect="<url>"` to the element to make the connection.

When designing your server application, remember that SSE works just like any HTTP request. Although you cannot send any
messages to the server after you have established a connection, you can send parameters to the server along with your
request. So, instead of making an SSE connection to your server at `https://my-server/chat-updates` you can also connect
to `https://my-server/chat-updates?friends=true&format=detailed`. This allows your server to customize its responses to
what your client needs.

### Receiving Named Events

SSE messages consist of an event name and a data packet. No other metadata is allowed in the message. Here is an
example:

```txt
event: EventName
data: <div>Content to swap into your HTML page.</div>
```

We'll use the `sse-swap` attribute to listen for this event and swap its contents into our webpage.

```html

<div hx-ext="sse" sse-connect="/event-source" sse-swap="EventName"></div>
```

Notice that the name `EventName` from the server's message must match the value in the `sse-swap` attribute. Your server
can use as many different event names as necessary, but be careful: browsers can only listen for events that have been
explicitly named. So, if your server sends an event named `ChatroomUpdate` but your browser is only listening for events
named `ChatUpdate` then the extra event will be discarded.

### Receiving Unnamed Events

SSE messages can also be sent without any event name. In this case, the browser uses the default name `message` in its
place. The same rules specified above still apply. If your server sends an unnamed message, then you must listen for it
by including `sse-swap="message"`. There is no option for using a catch-all name. Here's how this looks:

```txt
data: <div>Content to swap into your HTML page.</div>
```

```html

<div hx-ext="sse" sse-connect="/event-source" sse-swap="message"></div>
```

### Receiving Multiple Events

You can also listen to multiple events (named or unnamed) from a single EventSource. Listeners must be either 1) the
same element that contains the `hx-ext` and `sse-connect` attributes, or 2) child elements of the element containing
the `hx-ext` and `sse-connect` attributes.

```html

Multiple events in the same element
<div hx-ext="sse" sse-connect="/server-url" sse-swap="event1,event2"></div>

Multiple events in different elements (from the same source).
<div hx-ext="sse" sse-connect="/server-url">
    <div sse-swap="event1"></div>
    <div sse-swap="event2"></div>
</div>
```

### Trigger Server Callbacks

When a connection for server sent events has been established, child elements can listen for these events by using the
special [`hx-trigger`](https://htmx.org/attributes/hx-trigger) syntax `sse:<event_name>`. This, when combined with
an `hx-get` or similar will trigger the element to make a request.

Here is an example:

```html

<div hx-ext="sse" sse-connect="/event_stream">
    <div hx-get="/chatroom" hx-trigger="sse:chatter">
        ...
    </div>
</div>
```

This example establishes an SSE connection to the `event_stream` end point which then triggers
a `GET` to the `/chatroom` url whenever the `chatter` event is seen.

### Automatic Reconnection

If the SSE Event Stream is closed unexpectedly, browsers are supposed to attempt to reconnect automatically. However, in
rare situations this does not work and your browser can be left hanging. This extension adds its own reconnection
logic (using an [exponential-backoff algorithm](https://en.wikipedia.org/wiki/Exponential_backoff)) on top of the
browser's automatic reconnection, so that your SSE streams will always be as reliable as possible.

### Testing SSE Connections with the Demo Server

Htmx includes a demo SSE server written in Node.js that will help you to see SSE in action, and begin bootstrapping your
own SSE code. It is located in the /test/ws-sse folder of
the [`htmx-extensions`](https://github.com/bigskysoftware/htmx-extensions) repository. Look at /test/ws-sse/README.md
for instructions on running and using the test server.

### Migrating from Previous Versions

Previous versions of htmx used a built-in tag `hx-sse` to implement Server Sent Events. This code has been migrated into
an extension instead. Here are the steps you need to take to migrate to this version:

| Old Attribute                  | New Attribute            | Comments                                                                                                                                                                                        |
|--------------------------------|--------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `hx-sse=""`                    | `hx-ext="sse"`           | Use the `hx-ext="sse"` attribute to install the SSE extension into any HTML element.                                                                                                            |
| `hx-sse="connect:<url>"`       | `sse-connect="<url>"`    | Add a new attribute `sse-connect` to the tag that specifies the URL of the Event Stream.  This attribute must be in the same tag as the `hx-ext` attribute.                                     |
| `hx-sse="swap:<EventName>"`    | `sse-swap="<EventName>"` | Add a new attribute `sse-swap` to any elements that will be swapped in via the SSE extension.  This attribute must be placed **on** or **inside of** the tag containing the `hx-ext` attribute. |
| `hx-trigger="sse:<EventName>"` | NO CHANGE                | any `hx-trigger` attributes do not need to change.  The extension will identify these attributes and add listeners for any events prefixed with `sse:`                                          |

### Listening to events dispatched by this extension

This extension dispatches several events. You can listen for these events like so:

```javascript
document.body.addEventListener('htmx:sseBeforeMessage', function (e) {
    // do something before the event data is swapped in
})
```

Each event object has a `detail` field that contains details of the event.

#### `htmx:sseOpen`

This event is dispatched when an SSE connection has been successfully established.

##### Details

* `detail.elt` - The element on which the SSE connection was setup. This is the element which has the `sse-connect`
  attribute.
* `detail.source` - The [EventSource](https://developer.mozilla.org/en-US/docs/Web/API/EventSource) object.

#### `htmx:sseError`

This event is dispatched when an SSE connection could not be established.

##### Details

* `detail.error` - The error that occurred while creating
  an [EventSource](https://developer.mozilla.org/en-US/docs/Web/API/EventSource).
* `detail.source` - The [EventSource](https://developer.mozilla.org/en-US/docs/Web/API/EventSource).

#### `htmx:sseBeforeMessage`

This event is dispatched just before the SSE event data is swapped into the DOM. If you don't want to swap
call `preventDefault()` on the event. Additionally the `detail` field is
a [MessageEvent](https://developer.mozilla.org/en-US/docs/Web/API/EventSource/message_event) - this is the event created
by [EventSource](https://developer.mozilla.org/en-US/docs/Web/API/EventSource) when it receives an SSE message.

##### Details

* `detail.elt` - The swap target.

#### `htmx:sseMessage`

This event is dispatched after the SSE event data has been swapped into the DOM. The `detail` field is
a [MessageEvent](https://developer.mozilla.org/en-US/docs/Web/API/EventSource/message_event) - this is the event created
by [EventSource](https://developer.mozilla.org/en-US/docs/Web/API/EventSource) when it receives an SSE message.

#### `htmx:sseClose`

This event is dispatched in three different closing scenario. To control for the scenario the user can control for the
evt.detail.sseclose property.

```javascript
document.body.addEventListener('htmx:sseClose', function (e) {
    const reason = e.detail.type
    switch (reason) {
        case "nodeMissing":
            // Parent node is missing and therefore connection was closed
        ...
        case "nodeReplaced":
            // Parent node replacement caused closing of connection
        ...
        case "message":
            // connection was closed due to reception of message sse-close
        ...
    }
})
```

## SSE in FastHTML with HTMX 2

Server-sent events (SSE) allow a server to push new data to a web page at any time. Unlike WebSockets, SSE is **unidirectional** (server to client only) and is part of the HTTP specification.

### Complete Example

FastHTML provides several tools for working with SSE. Here's a complete example with detailed annotations:

```python
import random
from asyncio import sleep
from fasthtml.common import *

# 1. Import the HTMX SSE extension
hdrs = (Script(src="https://unpkg.com/htmx-ext-sse@2.2.1/sse.js"),)
app, rt = fast_app(hdrs=hdrs)

@rt
def index():
    return Titled("SSE Random Number Generator",
        P("Generate pairs of random numbers, as the list grows scroll downwards."),
        Div(
            hx_ext="sse",                    # 2. Tell HTMX to load the SSE extension
            sse_connect="/number-stream",    # 3. Look at /number-stream for SSE content
            hx_swap="beforeend show:bottom", # 4. Add new items at end and scroll down
            sse_swap="message"               # 5. Specify the event name
        )
    )

# 6. Set up the asyncio event loop
shutdown_event = signal_shutdown()

# 7. Don't forget to make this an async function!
async def number_generator():
    # 8. Iterate through the asyncio event loop
    while not shutdown_event.is_set():
        # 9. Yield data as FT components
        data = Article(random.randint(1, 100))
        yield sse_message(data)
        await sleep(1)

# 10. The endpoint must be async and return an EventStream
@rt("/number-stream")
async def get(): 
    return EventStream(number_generator())
```

### Key Points Explained

1. **Import the extension**: Add the HTMX SSE extension script to your app headers
2. **Load extension**: Use `hx_ext="sse"` on the container element
3. **Connect to endpoint**: `sse_connect` points to your SSE endpoint URL
4. **Configure swap**: `hx_swap="beforeend show:bottom"` adds items at the end and auto-scrolls
5. **Event name**: `sse_swap="message"` is FastHTML's default event name (only change if you have multiple SSE endpoints)
6. **Async setup**: Use `signal_shutdown()` for clean shutdown handling
7. **Async generator**: The generator function must be `async`
8. **Loop control**: Check `shutdown_event.is_set()` to handle graceful shutdown
9. **Yield FT components**: Data should be FastHTML components for seamless HTMX integration
10. **EventStream return**: The endpoint must return `EventStream(generator())`
